# 05 — Model Comparison & Error Analysis

This notebook:
1. Compares all three models side-by-side
2. Analyzes 25 misclassified samples from the best model
3. Categorizes failure modes
4. Generates category-level summaries using TextRank

Prerequisites: Run notebooks 02, 03, and 04 first.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import joblib
from pathlib import Path

from src.baseline import predict_baseline
from src.attention_model import load_vocab, predict_with_attention, BiLSTMAttention
from src.transformer_model import predict_transformer, load_transformer
from src.data_prep import compute_class_weights, score_priority
from src.evaluate import compare_models, collect_errors, plot_confusion_matrix

sns.set_theme(style='whitegrid')
BASE_DIR   = Path('..')
DATA_DIR   = BASE_DIR / 'data' / 'processed'
MODELS_DIR = BASE_DIR / 'models'
RESULTS_DIR = BASE_DIR / 'results'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Ready. Device:', DEVICE)

## 1. Load Data and All Models

In [ ]:
test_df = pd.read_csv(DATA_DIR / 'test.csv')
le_product = joblib.load(MODELS_DIR / 'label_encoder_product.joblib')
label_names = list(le_product.classes_)

test_texts_clean = test_df['narrative_clean'].tolist()
test_texts_raw   = test_df['Consumer complaint narrative'].fillna('').tolist()
y_test = test_df['product_encoded'].values

# Baseline
baseline_pipeline = joblib.load(MODELS_DIR / 'baseline_pipeline_product.joblib')

# BiLSTM
vocab = load_vocab(MODELS_DIR / 'vocab.json')
NUM_CLASSES = len(label_names)
bilstm_model = BiLSTMAttention(vocab_size=len(vocab), num_classes=NUM_CLASSES)
bilstm_model.load_state_dict(torch.load(MODELS_DIR / 'bilstm_attention.pt', weights_only=True))
bilstm_model.to(DEVICE)

# DistilBERT
distilbert_model, distilbert_tokenizer = load_transformer(task='product')

print('All models loaded.')

## 2. Generate Predictions from All Models

In [ ]:
print('Running baseline predictions...')
base_preds, base_probas = predict_baseline(baseline_pipeline, pd.Series(test_texts_clean))

print('Running BiLSTM predictions...')
bilstm_preds, bilstm_probas, _ = predict_with_attention(
    bilstm_model, test_texts_clean, vocab, max_len=256, device=DEVICE,
)

print('Running DistilBERT predictions...')
bert_preds, bert_probas = predict_transformer(
    distilbert_model, distilbert_tokenizer, test_texts_raw,
    max_length=256, batch_size=32, device=DEVICE,
)

print('All predictions complete.')

## 3. Load Saved Metrics & Build Comparison Table

In [ ]:
def load_metrics(filename):
    path = RESULTS_DIR / filename
    with open(path) as f:
        return json.load(f)

all_metrics = {
    'TF-IDF + LR (Baseline)':      load_metrics('baseline_product_metrics.json'),
    'BiLSTM + Attention':           load_metrics('bilstm_attention_product_metrics.json'),
    'DistilBERT (Fine-Tuned)':      load_metrics('distilbert_product_metrics.json'),
}

comparison_df = compare_models(all_metrics, label_names)
print('\nModel Comparison Table:')
comparison_df

In [ ]:
from IPython.display import Image
Image(str(RESULTS_DIR / 'model_comparison_bar.png'))

In [ ]:
Image(str(RESULTS_DIR / 'per_class_f1_comparison.png'))

## 4. Per-Model Confidence Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
model_probas = [base_probas, bilstm_probas, bert_probas]
model_names  = ['Baseline', 'BiLSTM+Attention', 'DistilBERT']
colors       = ['#4C72B0', '#DD8452', '#55A868']

for ax, probas, name, color in zip(axes, model_probas, model_names, colors):
    max_conf = probas.max(axis=1)
    ax.hist(max_conf, bins=30, color=color, edgecolor='white', alpha=0.8)
    ax.axvline(max_conf.mean(), color='black', linestyle='--', linewidth=1.5,
               label=f'Mean: {max_conf.mean():.2f}')
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Max Predicted Probability')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('Prediction Confidence Distribution per Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Error Analysis — Misclassified Samples

In [ ]:
# Analyze errors from the best model (DistilBERT)
errors_df = collect_errors(
    texts=test_texts_raw,
    y_true=y_test,
    y_pred=bert_preds,
    y_proba=bert_probas,
    label_names=label_names,
    model_name='distilbert_product',
    n_samples=25,
)

print(f'Error types breakdown:')
print(errors_df['error_type'].value_counts())
errors_df[['error_type', 'true_label', 'predicted_label', 'confidence', 'word_count', 'text_excerpt']].head(10)

In [ ]:
# Error type distribution pie chart
error_counts = errors_df['error_type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].pie(error_counts.values, labels=error_counts.index,
            autopct='%1.1f%%', startangle=90,
            colors=sns.color_palette('Set2', len(error_counts)))
axes[0].set_title('Error Type Distribution (DistilBERT)', fontweight='bold')

# Confusion pair analysis — which class pairs are most confused?
confusion_pairs = (
    errors_df.groupby(['true_label', 'predicted_label'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(15)
)
confusion_pairs['pair'] = confusion_pairs['true_label'].str[:15] + ' → ' + confusion_pairs['predicted_label'].str[:15]

axes[1].barh(confusion_pairs['pair'][::-1], confusion_pairs['count'][::-1],
             color='salmon', edgecolor='white')
axes[1].set_xlabel('Number of Errors')
axes[1].set_title('Most Confused Class Pairs', fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'error_analysis_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Deep Dive — Sample Misclassified Complaints

In [ ]:
print('=== Sample Misclassified Complaints (DistilBERT) ===\n')
for _, row in errors_df.head(10).iterrows():
    print(f'Error Type:   {row["error_type"]}')
    print(f'True Label:   {row["true_label"]}')
    print(f'Predicted:    {row["predicted_label"]} (confidence: {row["confidence"]:.2f})')
    print(f'Word Count:   {row["word_count"]}')
    print(f'Text:         {row["text_excerpt"]}')
    print('-' * 80)

## 7. Why Each Model Fails

### Baseline (TF-IDF + LR)
- **No word order:** "not a problem" has same features as "a problem" if split into unigrams.
- **Sparse features:** rare domain terms (e.g., CHEXSYSTEMS, OFAC) may be below min_df threshold.
- **No semantic similarity:** "fraud" and "scam" are treated as completely different features.

### BiLSTM + Attention
- **Fixed vocabulary:** OOV tokens default to `<UNK>`, losing signal from uncommon but important words.
- **No pretraining:** Limited understanding of financial domain concepts from scratch.
- **Sequential processing:** Distant tokens interact only through multiple LSTM steps.

### DistilBERT Failures
- **Truncation:** Long complaints (>256 tokens) lose tail content — sometimes the key issue is mentioned late.
- **Class overlap:** Credit Card vs Checking Account complaints often involve similar issues (fees, access).
- **Vague narratives:** Some complaints are too general to classify precisely even for humans.
- **Rare classes:** Low-resource categories (e.g., Payday Loan) may still be confused with Mortgage.

## 8. Summarization — Top Complaint Patterns per Category

In [ ]:
import nltk
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)

def summarize_complaints(texts, n_sentences=3, n_complaints=50):
    """TextRank extractive summary of a collection of complaint texts."""
    # Sample up to n_complaints, combine into one document
    sample = texts[:n_complaints]
    combined = ' '.join(str(t) for t in sample if isinstance(t, str))
    if len(combined.strip()) < 100:
        return 'Insufficient text for summarization.'
    try:
        parser = PlaintextParser.from_string(combined, Tokenizer('english'))
        summarizer = TextRankSummarizer()
        summary = summarizer(parser.document, n_sentences)
        return ' '.join(str(s) for s in summary)
    except Exception as e:
        return f'Summarization failed: {e}'

print('Summarizer ready.')

In [ ]:
# Load full filtered data
full_df = pd.read_csv(DATA_DIR / 'full_filtered.csv')

summaries = {}
print('=== Top Complaint Patterns per Category (TextRank) ===\n')

for product in sorted(full_df['Product'].unique()):
    subset_texts = full_df[full_df['Product'] == product]['Consumer complaint narrative'].tolist()
    summary = summarize_complaints(subset_texts, n_sentences=2, n_complaints=50)
    summaries[product] = summary
    print(f'[{product}]')
    print(summary)
    print()

with open(RESULTS_DIR / 'category_summaries.json', 'w') as f:
    json.dump(summaries, f, indent=2)
print('Summaries saved.')

## 9. Research Questions — Answers

**Q1: Can a fine-tuned Transformer outperform a simplified attention-based model?**
→ See comparison table above. DistilBERT is expected to outperform BiLSTM+Attention on Macro F1.

**Q2: Which categories are hardest/easiest to classify?**
→ See per-class F1 chart. Classes with distinctive vocabulary (e.g., Mortgage, Student Loan) are easiest.
   Credit Card vs Checking Account is typically the hardest confusion pair.

**Q3: Does attention improve over baseline?**
→ Compare Baseline vs BiLSTM F1 scores. The attention mechanism should help on ambiguous, longer texts.

**Q4: Where does the Transformer fail?**
→ Truncation of long narratives, vague complaints, class overlap between similar financial products.

**Q5: Can this system help prioritize complaints?**
→ Yes — the rule-based priority scorer flags fraud/security complaints as Critical/High with high recall.